In [ ]:
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from plotly.subplots import make_subplots
import plotly.graph_objs as go
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV
import shap
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
import plotly.express as px
from catboost import CatBoostRegressor
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.compose import TransformedTargetRegressor

In [2]:
df = pd.read_csv("../data/diamonds.csv")

In [3]:
df = df.drop_duplicates()

In [4]:
x_precentile_99 = df['x'].quantile(0.99)
y_precentile_99 = df['y'].quantile(0.99)
z_precentile_99 = df['z'].quantile(0.99)
x_precentile_01 = df['x'].quantile(0.01)
y_precentile_01 = df['y'].quantile(0.01)
z_precentile_01 = df['z'].quantile(0.01)
df = df[df['x']<= x_precentile_99]
df = df[df['y']<= y_precentile_99]
df = df[df['z']<= z_precentile_99]
df = df[df['x']>= x_precentile_01]
df = df[df['y']>= y_precentile_01]
df = df[df['z']>= z_precentile_01]

In [5]:
RANDOM_STATE = 12345

In [6]:
X = df.drop('price', axis = 1)

In [7]:
y = df['price']

In [8]:
X.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,x,y,z
3,4,0.29,Premium,I,VS2,62.4,58.0,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,4.34,4.35,2.75
7,8,0.26,Very Good,H,SI1,61.9,55.0,4.07,4.11,2.53
10,11,0.30,Good,J,SI1,64.0,55.0,4.25,4.28,2.73
13,14,0.31,Ideal,J,SI2,62.2,54.0,4.35,4.37,2.71


In [9]:
X = X.drop(columns = 'Unnamed: 0')

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)

In [11]:
X_train.shape

(42017, 9)

In [12]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()

In [13]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_48101/4180746908.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns.tolist()


In [14]:
cat_cols = [col for col in cat_cols if col not in ['cut', 'color', 'clarity']]

In [15]:
ord_cols = ['cut', 'color', 'clarity']

In [16]:
X_train['cut'].value_counts()

cut
Ideal        17060
Premium      10671
Very Good     9270
Good          3814
Fair          1202
Name: count, dtype: int64

In [17]:
X_train['color'].value_counts()

color
G    8895
E    7534
F    7482
H    6482
D    5357
I    4169
J    2098
Name: count, dtype: int64

In [18]:
X_train['clarity'].value_counts()

clarity
SI1     10357
VS2      9642
SI2      7004
VS1      6454
VVS2     3828
VVS1     2779
IF       1436
I1        517
Name: count, dtype: int64

In [19]:
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

In [20]:
preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),

    ('ord',
     OrdinalEncoder(
         categories=[
             cut_order,
             color_order,
             clarity_order
         ]
     ),
     ord_cols),

    ('cat',
     OneHotEncoder(handle_unknown='ignore'),
     cat_cols)
])

In [21]:
models = {
    "Dummy Regressor": DummyRegressor(strategy="mean"),

    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        verbose=0,
        random_state=RANDOM_STATE
    )
}

In [22]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

baseline_results = []

for model_name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    baseline_results.append({
        "model": model_name,
        "mae_mean": -scores["test_mae"].mean(),
        "mae_std": scores["test_mae"].std(),
        "rmse_mean": -scores["test_rmse"].mean(),
        "rmse_std": scores["test_rmse"].std(),
        "r2_mean": scores["test_r2"].mean(),
        "r2_std": scores["test_r2"].std()
    })

baseline_results = (
    pd.DataFrame(baseline_results)
    .sort_values("rmse_mean")
    .reset_index(drop=True)
)

baseline_results

,model,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
0,CatBoost,268.720529,5.568786,498.328725,18.377796,0.982785,0.000829
1,Random Forest,259.667255,7.559430,514.416804,27.146931,0.981645,0.001446
2,Decision Tree,345.576290,10.959375,692.964383,29.851934,0.966694,0.002213
3,Linear Regression,745.996976,5.786517,1096.324650,10.655277,0.916656,0.001486
4,Dummy Regressor,2892.609664,32.907618,3798.615866,51.479824,-0.000321,0.000249


In [23]:
best_model_name = baseline_results.loc[0, "model"]
best_model_name

best_model = models[best_model_name]

best_baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

best_baseline_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](0,)",[]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['carat','cut','color',...,'x','y','z']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not sp

In [24]:
y_pred = best_baseline_pipeline.predict(X_test)

test_metrics = {
    "mae": mean_absolute_error(y_test, y_pred),
    "rmse": np.sqrt(mean_squared_error(y_test, y_pred, )),
    "r2": r2_score(y_test, y_pred)
}

test_metrics_df = pd.DataFrame([test_metrics])

display(test_metrics_df)

,mae,rmse,r2
0,270.187135,510.708852,0.981826


In [25]:
predictions_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred
})

predictions_df["error"] = (
    predictions_df["y_true"] - predictions_df["y_pred"]
)

display(predictions_df.head())

,y_true,y_pred,error
16975,6779,6860.985507,-81.985507
19766,8334,8866.182606,-532.182606
18733,7652,7437.437820,214.562180
21030,625,670.379248,-45.379248
44058,1550,2004.591216,-454.591216


In [26]:
px.histogram(
    predictions_df,
    x="error",
    nbins=50,
    title="Распределение ошибок модели",
    labels={"error": "Ошибка прогноза"}
)

In [27]:
px.scatter(
    predictions_df.sample(5000, random_state=42),
    x="y_true",
    y="y_pred",
    opacity=0.5,
    title="Сравнение фактических и предсказанных значений",
    labels={
        "y_true": "Фактическая цена",
        "y_pred": "Предсказанная цена"
    }
)

Catboost оказался лучшей моделью с mae_mean 270.187135 и r2 0.981826

